# 00 Common setup (do not run on its own)

Every other notebook starts with `%run ./00_common.ipynb`. This notebook:

* sets the project paths,
* imports the InverseTHzSim code from `../InverseThzSim/src` in the same order as the authors' `Import_loader.py`,
* defines the parameter ranges, the scene builder, the feature function, the dataset loader and small helpers.

`FAKE_MODE = True` (set by the environment variable `KOLB_FAKE=1`) skips Mitsuba completely and makes random data, so notebooks 04 and 06 can be tested on a CPU before the GPU is free.

In [ ]:
import os, sys, json, time, math, glob, random
from pathlib import Path
import numpy as np

_default_project = (Path.home() / 'Documents' / 'Kolb_Radar_Project') if os.name == 'nt' else Path('/mnt/c/Users/Rey/Documents/Kolb_Radar_Project')
PROJECT = Path(os.environ.get('KOLB_PROJECT', str(_default_project))).resolve()
REPO    = PROJECT / 'InverseThzSim'
OUT     = PROJECT / 'outputs'
DATA    = OUT / 'data'
MODELS  = OUT / 'models'
FIGS    = OUT / 'figures'
RESULTS = OUT / 'results'
for p in (DATA, MODELS, FIGS, RESULTS):
    p.mkdir(parents=True, exist_ok=True)

FAKE_MODE = os.environ.get('KOLB_FAKE', '0') == '1'
SEED = 0
random.seed(SEED); np.random.seed(SEED)
print('PROJECT  :', PROJECT)
print('FAKE_MODE:', FAKE_MODE)

In [ ]:
# Import the authors' code exactly like notebooks/Import_loader.py does.
# Their code uses relative paths such as ../data/..., so the working directory must be REPO/notebooks.
if not FAKE_MODE:
    os.chdir(REPO / 'notebooks')
    if str(REPO / 'src') not in sys.path:
        sys.path.append(str(REPO / 'src'))
    from config import config
    import init_system
    # The authors built Mitsuba 3.3 from source with cuda_mono / cuda_ad_mono. The pip wheel of Mitsuba 3.3.0
    # (same version) ships cuda_ad_rgb / llvm_ad_rgb / scalar_rgb only, so the variant comes from KOLB_MI_VARIANT.
    MI_VARIANT = os.environ.get('KOLB_MI_VARIANT', 'cuda_ad_rgb')
    def _comp_grad_r(flag):
        import mitsuba as mi
        config['render_grad'] = flag
        init_system.mega_kernel(False)
        mi.set_variant(MI_VARIANT)
    init_system.comp_grad_r = _comp_grad_r
    init_system.comp_grad_r(False)
    import mitsuba as mi
    print('mitsuba variant:', mi.variant())
    import sampler as smp
    smp.init_RNG()
    import utils, sparams, scenes, measurement, losses, setup_utils
    import optimisation as opt
    sparams.mimo_system.switch_off([2], 10)  # same Tx/Rx masks as the paper
    config['material'] = 'phong'
    config['cal_strat'] = 'simulated'
    import torch
    DEVICE = config['device']
else:
    import torch
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device', DEVICE)

In [ ]:
# Parameter ranges for synthetic spheres (metres). They cover the real measured positions in src/sparams.py:
# x from -0.108 to 0.0955, y = 0.041, z from 0.320 to 0.345, radius from 0.01 to 0.02.
PARAM_NAMES = ['x', 'y', 'z', 'r']
LOW  = np.array([-0.120, 0.020, 0.310, 0.008], dtype=np.float32)
HIGH = np.array([ 0.120, 0.060, 0.350, 0.022], dtype=np.float32)

def sample_params(n, rng):
    return (LOW + (HIGH - LOW) * rng.random((n, 4))).astype(np.float32)

def to_unit(p):   return (p - LOW) / (HIGH - LOW)
def from_unit(u): return LOW + u * (HIGH - LOW)

T_IDX = list(range(1, 11))   # transmitters used by the authors' torch_fft_l2 loss

In [ ]:
def make_sphere_scene(x, y, z, r):
    """Same call order as setup_utils.exp_sphere_radius: set_position then set_radius."""
    s = scenes.OneSphere()
    s.set_position(float(x), float(y), float(z))
    s.set_radius(float(r))
    return s

def simulate(params):
    """Run the authors' simulator for one sphere and return a calibrated Measurement."""
    x, y, z, r = [float(v) for v in params]
    sc = make_sphere_scene(x, y, z, r)
    msr = measurement.Simulated(sc)   # Simulated expects the THZScene object (it calls get_name and get_scene)
    msr.calibrate(setup_utils.cal_path())
    return msr

def featurise_complex(cpx):
    """Phase robust features, same idea as losses.torch_fft_l2:
    normalise, Hamming window over frequency, FFT, magnitude. Output float32 [10, N_Rx, 64]."""
    cpx = torch.as_tensor(cpx)
    cpx = cpx / (cpx.abs().max() + 1e-12)
    cpx = cpx[T_IDX]
    w = torch.hamming_window(cpx.shape[-1], device=cpx.device)
    f = torch.fft.fft(cpx * w).abs()
    f = f / (f.max() + 1e-12)
    return f.float().cpu().numpy()

def featurise(msr):
    return featurise_complex(msr.get_measurement())

In [ ]:
FEAT_SHAPE = (10, 31, 64)

def fake_features(params):
    """CPU test data: a smooth fake signal that depends on the parameters (only for testing code)."""
    u = to_unit(params)
    t = np.arange(10)[:, None, None]; rr = np.arange(31)[None, :, None]; f = np.arange(64)[None, None, :]
    peak = 10 + 40 * u[2] + 5 * np.sin(t * 0.3 + u[0] * 6) + 3 * u[1] * np.cos(rr * 0.2)
    sig = np.exp(-((f - peak) ** 2) / (2 * (1.5 + 3 * u[3]) ** 2))
    return (sig + 0.02 * np.random.randn(*sig.shape)).astype(np.float32)

def load_dataset():
    shards = sorted(DATA.glob('shard_*.npz'))
    if not shards:
        raise FileNotFoundError('No shards in outputs/data. Run 03 first, or set KOLB_FAKE=1.')
    X, P = [], []
    for s in shards:
        d = np.load(s); X.append(d['X'].astype(np.float32)); P.append(d['P'])
    return np.concatenate(X), np.concatenate(P)

def split_indices(n, seed=SEED):
    idx = np.random.default_rng(seed).permutation(n)
    n_te = max(1, int(0.05 * n)); n_va = max(1, int(0.05 * n))
    return idx[n_te + n_va:], idx[n_te:n_te + n_va], idx[:n_te]

def save_json(obj, name):
    path = RESULTS / name
    with open(path, 'w') as f:
        json.dump(obj, f, indent=2)
    print('saved', path)

def log(msg):
    line = time.strftime('%Y-%m-%d %H:%M:%S') + '  ' + msg
    print(line)
    with open(PROJECT / 'LOG.md', 'a') as f:
        f.write('* ' + line + '\n')

In [ ]:
def run_inverse_1d(gt_params, x_init, epochs=50, nsamples=1, sigma=0.0125, lr=2e-2, tol_mm=1.0, verbose=False):
    """Same algorithm as optimisation.run_optimization, but it also records the error per iteration."""
    x_t, y_t, z_t, r_t = [float(v) for v in gt_params]
    gt_msr = simulate(gt_params)
    scene = scenes.OneSphere(); scene.set_position(0.0, y_t, z_t); scene.set_radius(r_t)
    update_fn = scene.translate_x
    ctx = {'scene': scene, 'init_pos': [x_init], 'gt_msr': gt_msr, 'update_fn': update_fn,
           'loss_fn': losses.torch_fft_l2, 'sampler': 'importance', 'antithetic': True,
           'nsamples': nsamples, 'sigma': sigma, 'device': config['device']}
    theta = torch.tensor([x_init], requires_grad=True, device=config['device'])
    optim = torch.optim.Adam([theta], lr=lr)
    smooth = opt.smoothFn(opt.sim_smooth, context_args=None, device=config['device'])
    hist, t0, hit = [abs(x_init - x_t) * 1000], time.time(), None
    best, best_loss = x_init, float('inf')
    for j in range(epochs):
        optim.zero_grad()
        loss = smooth(theta.unsqueeze(0), ctx); loss.backward(); optim.step()
        with torch.no_grad():
            cur = opt.get_mts_measurements(theta, update_fn, ctx)
            l = ctx['loss_fn'](cur, gt_msr).item()
            if l < best_loss: best_loss, best = l, float(theta.item())
            err = abs(float(theta.item()) - x_t) * 1000
            hist.append(err)
            if hit is None and err < tol_mm: hit = j + 1
        if verbose: print(f'iter {j+1}: x={theta.item():.5f}  err={err:.2f} mm  loss={l:.4g}')
    return {'final_err_mm': abs(best - x_t) * 1000, 'iters_to_tol': hit, 'seconds': time.time() - t0,
            'history_mm': hist, 'x_init': x_init, 'x_true': x_t}

In [ ]:
import torch.nn as nn

class OneShotNet(nn.Module):
    """Small CNN: raw radar features [10, 31, 64] -> sphere parameters in unit range [x, y, z, r]."""
    def __init__(self, c=(32, 64, 128), hidden=256):
        super().__init__()
        layers, cin = [], 10
        for co in c:
            layers += [nn.Conv2d(cin, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU(inplace=True), nn.MaxPool2d(2)]
            cin = co
        self.features = nn.Sequential(*layers)
        with torch.no_grad():
            n = self.features(torch.zeros(1, *FEAT_SHAPE)).numel()
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(n, hidden), nn.ReLU(inplace=True),
                                  nn.Dropout(0.1), nn.Linear(hidden, 4), nn.Sigmoid())
    def forward(self, x):
        return self.head(self.features(x))

def mae_mm(pred_unit, true_params):
    pred = from_unit(pred_unit)
    err = np.abs(pred - true_params) * 1000.0
    return {k: float(err[:, i].mean()) for i, k in enumerate(PARAM_NAMES)}

def evaluate(model, X, P, bs=256):
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(X), bs):
            out.append(model(torch.from_numpy(X[i:i+bs]).to(DEVICE)).cpu().numpy())
    return mae_mm(np.concatenate(out), P)
print('common ready')